# Лабораторная работа №1
## Предобработка текстовых данных для анализа судебных документов

**Дисциплина:** Информационно-поисковые системы  


---

## Введение

Настоящая лабораторная работа является первым практическим этапом в освоении методов построения информационно-аналитических систем (ИАС) для работы с юридической документацией. В основе любой системы анализа текстов лежит конвейер предобработки данных — последовательность операций, преобразующих «сырой» неструктурированный текст в форму, пригодную для применения алгоритмов машинного обучения и лингвистического анализа.

В ходе данной работы самостоятельно реализуйте полный конвейер предобработки на примере корпуса судебных актов, а также применит методы распознавания именованных сущностей (NER) и ключевых слов — фундаментальные инструменты любой современной ИАС. Полученный конвейер будет использован в последующих лабораторных работах при решении задач классификации документов и интеграции с большими языковыми моделями.

---

## Цель работы

Освоить методы сбора, очистки и лингвистической обработки текстовых данных применительно к корпусу судебных документов; получить практические навыки применения библиотек Python для токенизации, лемматизации, удаления стоп-слов и извлечения именованных сущностей.

---

## Задачи лабораторной работы

В ходе выполнения работы требуется решить следующие задачи в строгой последовательности:

1. Формирование учебного корпуса судебных документов.
2. Реализация функций очистки текста от артефактов форматирования.
3. Токенизация, нормализация регистра и удаление стоп-слов.
4. Морфологическая нормализация (лемматизация) с применением библиотеки `pymorphy2`.
5. Извлечение именованных сущностей с применением библиотеки `natasha`.
6. Ключевое-словарная классификация документов на основе разработанного конвейера.
7. Количественный анализ корпуса и визуализация результатов.

---

## Теоретическая справка

Обратитесь к материалам **Лекции 1** и **Лекции 2** перед выполнением практической части. Ключевые понятия, которые потребуются в работе, изложены ниже в краткой форме.

**Предобработка данных** — совокупность методов и техник, направленных на приведение исходных текстовых данных в форму, пригодную для обучения моделей или проведения аналитики. Качество предобработки напрямую определяет качество итоговой системы.

**Токенизация** — процесс разбиения текста на минимальные смысловые единицы (токены): слова, знаки препинания, числа. Является обязательным первым шагом практически любого NLP-конвейера.

**Лемматизация** — приведение слова к его нормальной (словарной) форме с учётом морфологических свойств. Например: «обрабатываются» → «обрабатывать». Для русскоязычных текстов используется библиотека `pymorphy2`.

**Стоп-слова** — частотные служебные слова (предлоги, союзы, местоимения), не несущие самостоятельного смысла и, как правило, удаляемые при обработке текста.

**Распознавание именованных сущностей (NER, Named Entity Recognition)** — задача выделения из текста упоминаний реальных объектов (персон, организаций, дат, мест) и их классификации по типам. В рамках данной работы применяется библиотека `natasha`, специализированная для обработки русскоязычных текстов.

---
## Часть 0. Установка и импорт зависимостей

Выполните следующую ячейку для установки необходимых библиотек. После установки может потребоваться перезапуск ядра (`Kernel → Restart`).

In [ ]:
# Установка зависимостей (выполните один раз)
!pip install pymorphy2 natasha nltk matplotlib wordcloud --quiet
!pip install pymorphy2-dicts-ru --quiet

: 

In [ ]:
import re
import string
import json
from collections import Counter

import nltk
import pymorphy2
import matplotlib.pyplot as plt
from wordcloud import WordCloud

from natasha import (
    Doc, Segmenter, NewsEmbedding,
    NewsNERTagger, NewsMorphTagger,
    MorphVocab
)

# Загрузка ресурсов NLTK
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

print('Все зависимости успешно загружены.')

---
## Часть 1. Формирование учебного корпуса

В рамках данной лабораторной работы используется учебный корпус, составленный из фрагментов публично доступных судебных актов арбитражных судов Российской Федерации. Каждый документ представлен в виде словаря (dict) с полями `id`, `category` и `text`.

В реальных условиях корпус формируется автоматически через парсинг открытых баз данных (например, kad.arbitr.ru), что подробно рассматривается в Лекции 2. В данной работе корпус задан статически для обеспечения воспроизводимости результатов.

**Задание 1.1.** Изучите структуру предложенного корпуса. Ответьте на вопросы: сколько документов содержится в корпусе? Какие категории представлены? Каков средний объём текста (в символах) по каждой категории?

In [ ]:
# Учебный корпус судебных актов
corpus = [
    {
        "id": "A40-001-2023",
        "category": "Оспаривание сделок должника",
        "text": (
            "Арбитражный суд г. Москвы рассмотрел заявление конкурсного управляющего "
            "ООО 'Строй-Инвест' Иванова А.П. о признании недействительной сделки — "
            "договора купли-продажи нежилых помещений от 15.03.2021, заключённого между "
            "должником и ООО 'Партнёр'. В соответствии со ст. 61.2 Федерального закона "
            "№ 127-ФЗ 'О несостоятельности (банкротстве)' суд установил, что сделка "
            "совершена в ущерб интересам кредиторов. Должник на дату совершения сделки "
            "отвечал признакам неплатёжеспособности. Рыночная стоимость имущества "
            "составила 42 500 000 рублей, тогда как цена сделки — 12 000 000 рублей. "
            "Суд признал договор недействительным и обязал ООО 'Партнёр' возвратить "
            "имущество в конкурсную массу должника."
        )
    },
    {
        "id": "A40-002-2023",
        "category": "Оспаривание сделок должника",
        "text": (
            "Определением Арбитражного суда Московской области удовлетворено заявление "
            "финансового управляющего Петрова Д.С. об оспаривании договора дарения "
            "квартиры, расположенной по адресу: Московская обл., г. Химки, ул. Ленина, д. 10. "
            "Договор заключён между гражданином-должником Сидоровым В.И. и его супругой "
            "Сидоровой О.Н. Согласно ст. 61.2 и ст. 61.3 Закона о банкротстве, сделка "
            "признана подозрительной: на момент её совершения должник имел непогашенную "
            "задолженность перед кредиторами на сумму свыше 8 000 000 рублей. "
            "Применены последствия недействительности сделки."
        )
    },
    {
        "id": "A56-003-2023",
        "category": "Субсидиарная ответственность",
        "text": (
            "Арбитражный суд г. Санкт-Петербурга и Ленинградской области рассмотрел "
            "заявление о привлечении к субсидиарной ответственности бывшего "
            "генерального директора ЗАО 'Балтик Трейд' Кузнецова М.В. "
            "В ходе судебного разбирательства установлено, что ответчик в нарушение "
            "ст. 61.11 Закона о банкротстве не передал конкурсному управляющему "
            "бухгалтерскую и иную документацию общества, что существенно затруднило "
            "формирование конкурсной массы. Размер субсидиарной ответственности "
            "составил 115 000 000 рублей. Заявление удовлетворено в полном объёме."
        )
    },
    {
        "id": "A56-004-2023",
        "category": "Субсидиарная ответственность",
        "text": (
            "Девятый арбитражный апелляционный суд оставил в силе определение "
            "суда первой инстанции о привлечении к субсидиарной ответственности "
            "участников ООО 'НордГрупп' — Романова П.А. и Васильевой Т.К. "
            "Судом установлено, что контролирующие должника лица довели организацию "
            "до банкротства посредством систематического вывода активов в пользу "
            "аффилированных структур в период с 2019 по 2021 год. "
            "На основании п. 1 ст. 61.12 ФЗ № 127 суд взыскал солидарно с ответчиков "
            "задолженность перед кредиторами в размере 78 300 000 рублей."
        )
    },
    {
        "id": "A60-005-2023",
        "category": "Обжалование налоговых решений",
        "text": (
            "ООО 'УралПром' обратилось в Арбитражный суд Свердловской области с "
            "заявлением о признании недействительным решения Межрайонной ИФНС России "
            "№ 25 по Свердловской области о доначислении налога на добавленную стоимость "
            "в размере 23 400 000 рублей и начислении пеней. Налоговый орган указал на "
            "неправомерное применение налоговых вычетов по операциям с контрагентами, "
            "обладающими признаками технических компаний. Суд, проанализировав "
            "представленные доказательства, установил реальность хозяйственных операций "
            "и отсутствие умысла налогоплательщика, в связи с чем заявление удовлетворил."
        )
    },
    {
        "id": "A60-006-2023",
        "category": "Обжалование налоговых решений",
        "text": (
            "Федеральный арбитражный суд Уральского округа рассмотрел кассационную "
            "жалобу ПАО 'Металлург' на постановление апелляционного суда, оставившего "
            "в силе решение о доначислении налога на прибыль организаций в размере "
            "67 000 000 рублей. Основанием для доначисления послужило исключение из "
            "состава расходов затрат по договорам с взаимозависимыми лицами. "
            "ФАС Уральского округа жалобу отклонил, подтвердив обоснованность применения "
            "налоговым органом методов трансфертного ценообразования в соответствии "
            "с разделом V.1 Налогового кодекса Российской Федерации."
        )
    },
    {
        "id": "A07-007-2023",
        "category": "Включение в реестр требований кредиторов",
        "text": (
            "В рамках дела о банкротстве ООО 'АгроЭкспресс' ПАО 'Агробанк' "
            "обратилось с заявлением о включении требований на сумму 34 200 000 рублей "
            "в реестр требований кредиторов должника. Требование основано на кредитном "
            "договоре № КД-2019/445 от 10.07.2019. Арбитражный суд Республики Башкортостан "
            "признал задолженность обоснованной и включил требования ПАО 'Агробанк' "
            "в третью очередь реестра требований кредиторов как обеспеченные залогом "
            "имущества должника."
        )
    },
    {
        "id": "A07-008-2023",
        "category": "Включение в реестр требований кредиторов",
        "text": (
            "Арбитражный суд Республики Татарстан отказал в удовлетворении заявления "
            "гражданина Фёдорова Н.Г. о включении требований на сумму 5 000 000 рублей "
            "в реестр требований кредиторов ЗАО 'ТатСтрой'. Суд установил, что требования "
            "заявителя носят корпоративный характер, поскольку основаны на договоре займа, "
            "заключённом между аффилированными лицами в условиях имущественного кризиса "
            "должника. В соответствии с разъяснениями Верховного Суда РФ, изложенными "
            "в Обзоре судебной практики № 4 (2021), такие требования подлежат "
            "понижению в очерёдности."
        )
    },
    {
        "id": "A40-009-2023",
        "category": "Оспаривание сделок должника",
        "text": (
            "Суд признал недействительным соглашение о зачёте встречных однородных "
            "требований, заключённое между ООО 'ТехноЛинк' и кредитором Смирновым Е.О. "
            "за три месяца до принятия заявления о банкротстве. Согласно ст. 61.3 "
            "Закона о банкротстве, данная сделка влечёт за собой предпочтительное "
            "удовлетворение требований одного кредитора перед другими. "
            "Конкурсному управляющему поручено подать заявление о взыскании "
            "неосновательного обогащения в размере 9 750 000 рублей."
        )
    },
    {
        "id": "A40-010-2023",
        "category": "Субсидиарная ответственность",
        "text": (
            "В рамках дела о несостоятельности ПАО 'СтройКомплекс' суд рассмотрел "
            "заявление об установлении оснований для привлечения к субсидиарной "
            "ответственности членов совета директоров: Белова Г.С., Орловой А.Д. "
            "и Нечаева В.П. Основание — одобрение заведомо убыточных сделок "
            "в 2020–2022 годах. Суд выделил требования в отдельное производство "
            "для определения точного размера ответственности после завершения "
            "формирования конкурсной массы."
        )
    }
]

print(f"Корпус загружен. Количество документов: {len(corpus)}")

In [ ]:
# ---- Задание 1.1 ----
# TODO: Выведите количество документов по каждой категории.
# TODO: Вычислите средний объём текста (в символах) по каждой категории.
# Используйте Counter из модуля collections и стандартные функции Python.

# Ваш код здесь:


---
## Часть 2. Очистка текста

Судебные документы, полученные из открытых источников, как правило, содержат HTML-разметку, специальные символы, лишние пробелы и иные артефакты. Перед проведением лингвистического анализа необходимо выполнить очистку текста.

Ниже приведены функции-заготовки. Некоторые из них содержат намеренно неполную реализацию — её необходимо дополнить.

**Задание 2.1.** Изучите предложенные функции очистки. Дополните функцию `normalize_whitespace` так, чтобы она удаляла повторяющиеся пробелы, табуляции и переносы строк, приводя текст к единой нормализованной форме.

**Задание 2.2.** Реализуйте функцию `remove_punctuation`, которая удаляет знаки пунктуации, сохраняя при этом дефисы внутри слов (пример: «купли-продажи» должно остаться нетронутым).

In [ ]:
def remove_html_tags(text: str) -> str:
    """Удаляет HTML-теги из текста."""
    clean = re.sub(r'<[^>]+>', '', text)
    return clean


def remove_special_chars(text: str) -> str:
    """Удаляет спецсимволы, оставляя буквы, цифры и базовую пунктуацию."""
    clean = re.sub(r'[^\w\s\-.,!?:;()"\'/]', '', text, flags=re.UNICODE)
    return clean


def normalize_whitespace(text: str) -> str:
    """Нормализует пробельные символы."""
    # TODO: реализуйте функцию — замените все последовательности пробельных
    # символов (\s+) единственным пробелом, удалите пробелы в начале и конце.
    pass


def remove_punctuation(text: str) -> str:
    """Удаляет знаки пунктуации, сохраняя дефисы внутри слов."""
    # TODO: реализуйте функцию.
    # Подсказка: сначала замените дефисы-разделители (не находящиеся между буквами)
    # на пробел, затем удалите оставшуюся пунктуацию.
    pass


def clean_text(text: str) -> str:
    """Полный конвейер очистки текста."""
    text = remove_html_tags(text)
    text = remove_special_chars(text)
    text = normalize_whitespace(text)
    return text.lower()


# Проверка на примере
sample = corpus[0]['text']
cleaned = clean_text(sample)
print("Оригинал (первые 200 символов):")
print(sample[:200])
print("\nПосле очистки (первые 200 символов):")
print(cleaned[:200])

---
## Часть 3. Токенизация и удаление стоп-слов

После очистки текст разбивается на токены — отдельные слова. Стоп-слова исключаются, поскольку они не несут содержательной информации для задач классификации и поиска.

**Задание 3.1.** Дополните список стоп-слов словами, характерными для юридических документов (например: «суд», «решение», «арбитражный», «рассмотрев», «установил» — слова, встречающиеся в большинстве судебных актов вне зависимости от их категории). Обоснуйте свой выбор в комментарии.

**Задание 3.2.** Примените функцию `tokenize_and_filter` ко всему корпусу. Убедитесь, что результат представляет собой список токенов для каждого документа.

In [ ]:
# Базовый набор стоп-слов для русского языка
STOP_WORDS = set(stopwords.words('russian'))

# TODO (Задание 3.1): добавьте в STOP_WORDS юридические стоп-слова
# Пример: STOP_WORDS.update(['суд', 'решение', ...])
LEGAL_STOPWORDS = set()  # Дополните этот набор
STOP_WORDS.update(LEGAL_STOPWORDS)


def tokenize_and_filter(text: str, stop_words: set = STOP_WORDS) -> list:
    """
    Выполняет токенизацию очищенного текста и фильтрацию стоп-слов.
    Возвращает список токенов длиной не менее 3 символов.
    """
    tokens = word_tokenize(text, language='russian')
    tokens = [
        token for token in tokens
        if token.isalpha() and len(token) >= 3 and token not in stop_words
    ]
    return tokens


# TODO (Задание 3.2): примените конвейер ко всему корпусу.
# Результат сохраните в переменную tokenized_corpus — список словарей,
# каждый из которых содержит поля 'id', 'category' и 'tokens'.
tokenized_corpus = []

# Ваш код здесь:


# Проверка
if tokenized_corpus:
    print(f"Документ {tokenized_corpus[0]['id']}")
    print(f"Категория: {tokenized_corpus[0]['category']}")
    print(f"Первые 15 токенов: {tokenized_corpus[0]['tokens'][:15]}")

---
## Часть 4. Лемматизация с использованием pymorphy2

Лемматизация приводит слова к их нормальной словарной форме. Это критически важно для русскоязычных текстов, поскольку богатая морфология русского языка порождает множество словоформ одного корня (например: «банкротства», «банкротством», «банкротству» → «банкротство»). Без лемматизации модель воспринимает эти формы как разные слова, что снижает качество анализа.

**Задание 4.1.** Реализуйте функцию `lemmatize_tokens`, используя `pymorphy2.MorphAnalyzer`. Примените её к `tokenized_corpus`.

**Задание 4.2.** Сравните частотные словари до и после лемматизации для одного из документов. Продемонстрируйте, что лемматизация объединяет словоформы одного слова.

In [ ]:
morph = pymorphy2.MorphAnalyzer()


def lemmatize_tokens(tokens: list) -> list:
    """
    Приводит список токенов к нормальной форме с помощью pymorphy2.
    Возвращает список лемм.
    """
    # TODO: для каждого токена получите его нормальную форму через morph.parse(token)[0].normal_form
    pass


# TODO (Задание 4.1): добавьте поле 'lemmas' в каждый документ tokenized_corpus
# Ваш код здесь:


# TODO (Задание 4.2): постройте и сравните Counter для tokens и lemmas первого документа.
# Выведите топ-10 наиболее частотных единиц в каждом случае.
# Ваш код здесь:


---
## Часть 5. Распознавание именованных сущностей (NER)

Извлечение именованных сущностей — одна из ключевых задач в ИАС для юридической сферы. Из судебных актов необходимо автоматически выделять упоминания организаций (стороны дела), персон (судьи, управляющие, ответчики), дат и денежных сумм.

В данной части применяется библиотека `natasha`, разработанная специально для обработки русскоязычных текстов новостного и документального стилей.

**Задание 5.1.** Изучите вывод функции `extract_entities`. Какие типы сущностей распознаёт модель? Насколько точно выделяются организации в судебных текстах? Зафиксируйте наблюдения в ячейке Markdown ниже.

**Задание 5.2.** Примените функцию ко всему корпусу. Постройте сводную таблицу наиболее часто встречающихся организаций (`ORG`) и персон (`PER`) по всему корпусу.

In [ ]:
# Инициализация компонентов natasha
segmenter = Segmenter()
emb = NewsEmbedding()
ner_tagger = NewsNERTagger(emb)
morph_tagger = NewsMorphTagger(emb)
morph_vocab = MorphVocab()


def extract_entities(text: str) -> list:
    """
    Извлекает именованные сущности из текста.
    Возвращает список кортежей (текст_сущности, тип_сущности).
    """
    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_ner(ner_tagger)

    entities = []
    for span in doc.spans:
        entities.append((span.text, span.type))
    return entities


# Пример на первом документе
sample_entities = extract_entities(corpus[0]['text'])
print(f"Именованные сущности в документе {corpus[0]['id']}:")
for entity_text, entity_type in sample_entities:
    print(f"  [{entity_type}] {entity_text}")

In [ ]:
# TODO (Задание 5.2):
# 1. Примените extract_entities ко всем документам корпуса.
# 2. Соберите отдельные Counter для типов PER (персоны) и ORG (организации).
# 3. Выведите топ-5 наиболее часто встречающихся сущностей каждого типа.

all_persons = Counter()
all_orgs = Counter()

# Ваш код здесь:


print("Наиболее частые организации:")
# Ваш код здесь:

print("\nНаиболее частые персоны:")
# Ваш код здесь:

**Задание 5.1 — Ваши наблюдения:**

> *Опишите здесь точность распознавания, ошибки модели (если обнаружены), типы распознаваемых сущностей.*

---
## Часть 6. Ключевое-словарная классификация документов

На основе результатов предобработки реализуется первый классификатор — правило-ориентированная система на базе ключевых слов. Несмотря на очевидную простоту, данный подход широко применяется в юридических ИАС как базовый метод (baseline), с которым сравниваются более сложные алгоритмы.

**Задание 6.1.** Изучите предложенный словарь ключевых слов. При необходимости расширьте его, добавив не менее трёх дополнительных лемм для каждой категории (обоснуйте выбор).

**Задание 6.2.** Вычислите точность классификатора на данном корпусе. Для каких категорий классификатор работает хуже? Предложите гипотезу, объясняющую ошибки.

In [ ]:
# Словарь ключевых лемм для каждой категории
KEYWORD_DICT = {
    "Оспаривание сделок должника": [
        "сделка", "недействительный", "оспаривание",
        "подозрительный", "предпочтение", "зачёт",
        # TODO: добавьте дополнительные леммы
    ],
    "Субсидиарная ответственность": [
        "субсидиарный", "ответственность", "контролировать",
        "генеральный", "директор", "довести",
        # TODO: добавьте дополнительные леммы
    ],
    "Обжалование налоговых решений": [
        "налог", "доначисление", "вычет",
        "ифнс", "налогоплательщик", "трансфертный",
        # TODO: добавьте дополнительные леммы
    ],
    "Включение в реестр требований кредиторов": [
        "реестр", "требование", "кредитор",
        "очередь", "включение", "банк",
        # TODO: добавьте дополнительные леммы
    ]
}


def classify_by_keywords(lemmas: list, keyword_dict: dict) -> str:
    """
    Классифицирует документ по набору лемм на основе словаря ключевых слов.
    Возвращает название категории с наибольшим числом совпадений.
    Если совпадений нет — возвращает 'Не определено'.
    """
    scores = {}
    lemmas_set = set(lemmas)
    for category, keywords in keyword_dict.items():
        scores[category] = len(lemmas_set.intersection(set(keywords)))

    best_category = max(scores, key=scores.get)
    return best_category if scores[best_category] > 0 else "Не определено"


# TODO (Задание 6.2): вычислите точность классификатора.
# Сравните предсказанные метки с истинными (поле 'category' в corpus).
# Выведите результаты для каждого документа: [ID] Истинная → Предсказанная (✓/✗)

correct = 0
total = len(tokenized_corpus)

# Ваш код здесь:


# print(f"\nТочность классификатора: {correct}/{total} = {correct/total:.1%}")

---
## Часть 7. Визуализация результатов

Количественный анализ и визуализация корпуса — обязательная часть любого аналитического исследования. По итогам предобработки необходимо продемонстрировать полученные результаты.

**Задание 7.1.** Постройте диаграмму (столбчатую или круговую), показывающую распределение документов по категориям в корпусе.

**Задание 7.2.** Постройте облако слов (WordCloud) для всего корпуса на основе лемматизированных токенов. Убедитесь, что в облаке не присутствуют стоп-слова.

**Задание 7.3 (повышенная сложность).** Постройте отдельные облака слов для каждой категории. Проанализируйте, насколько лексика разных категорий пересекается. Как это влияет на работу словарного классификатора из Части 6?

In [ ]:
# TODO (Задание 7.1): диаграмма распределения категорий
# Используйте matplotlib. Подпишите оси и добавьте заголовок.

# Ваш код здесь:


In [ ]:
# TODO (Задание 7.2): облако слов по всему корпусу
# Объедините все леммы из tokenized_corpus в одну строку через пробел.
# Создайте объект WordCloud с параметрами: font_path (для кириллицы),
# width=800, height=400, background_color='white'.

# Ваш код здесь:


In [ ]:
# TODO (Задание 7.3): отдельные облака слов по категориям

# Ваш код здесь:


---
## Контрольные вопросы

Ответьте развёрнуто (3–5 предложений) на каждый вопрос в отдельных ячейках Markdown.

1. Чем лемматизация отличается от стемминга? В каких сценариях предпочтительнее использование каждого из методов применительно к юридическим текстам?

2. Почему удаление стоп-слов улучшает качество классификации текстов? Можно ли привести пример, когда удаление стоп-слов было бы вредным?

3. Каковы принципиальные ограничения ключевое-словарного классификатора, реализованного в Части 6? Какими методами, рассмотренными в Лекции 2, можно преодолеть эти ограничения?

4. Почему для русскоязычных юридических текстов предпочтительно использование специализированных библиотек (таких как `natasha`) вместо универсальных решений (таких как spaCy с моделью для русского языка)?

**Ответ на вопрос 1:**

> *Ваш ответ здесь.*

**Ответ на вопрос 2:**

> *Ваш ответ здесь.*

**Ответ на вопрос 3:**

> *Ваш ответ здесь.*

**Ответ на вопрос 4:**

> *Ваш ответ здесь.*

---
## Индивидуальное задание

Самостоятельно добавьте в корпус **не менее 5 новых документов**, охватывающих не менее двух различных категорий. Тексты должны быть составлены на основе реальных судебных актов (открытые базы данных: kad.arbitr.ru, sudrf.ru) либо быть приближены к ним по стилю и терминологии.

После добавления документов:

1. Повторно запустите весь конвейер предобработки.
2. Пересчитайте точность ключевое-словарного классификатора.
3. Проанализируйте, как изменились результаты. Если точность снизилась, предложите, каким образом можно улучшить словарь ключевых слов.

Результаты оформите в итоговой ячейке Markdown с кратким аналитическим выводом (не менее 150 слов).